In [ ]:
import pandas as pd # The primary library for structured data manipulation and analysis.
import requests # Tool utilized for web communication.
from bs4 import BeautifulSoup # Tool for HTML parsing during data extraction.
import time # For rate limiting between requests.
import re # For pattern matching and string cleaning.
import numpy as np # For high-performance numerical operations.
import json # For handling and parsing metadata files.
import os # For filesystem operations and directory management.
import sys # For system-level interactions.

Takeaway : Successful import of all required libraries. No error messages should be displayed.

---

## Global Constants and URL Configuration

#### Purpose
Centralizing configuration parameters ensures the script remains maintainable. We include a User-Agent header to ensure successful communication with the host server.

In [ ]:
BASE_URL = "https://www.globalfirepower.com/"
LISTING_URL = BASE_URL + "countries-listing.php"
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

print("Global configuration constants and headers have been successfully initialized.")

Takeaway : Confirmation message that constants have been initialized and headers are set for ethical scraping.

---

### Load Initial Data

#### Purpose
Loading the cleaned dataset produced in Milestone 1.

In [ ]:
CLEANED_DATA_PATH = "data/cleaned/military_cleaned_data.csv"

if os.path.exists(CLEANED_DATA_PATH):
    df_cleaned = pd.read_csv(CLEANED_DATA_PATH)
    print(f"Successfully loaded cleaned dataset with {len(df_cleaned)} records.")
    print(df_cleaned.head())
else:
    print(f"CRITICAL: Cleaned data file NOT found at {CLEANED_DATA_PATH}. Please run Milestone 1 notebook first.")

Takeaway : Cleaned data is loaded into memory, providing the foundation for analytical engineering.

---

<h2 style="color: #117A65;">KPI Engineering and Tableau Prep</h2>

This section focuses on feature engineering and planning for data visualization.

## KPI Feature Engineering

### Purpose
Transforms raw metrics into strategic ratios. This code includes robust handling for missing keys and division by zero.

In [ ]:
def calculate_official_kpis(df, metadata_path):
    if df.empty: return pd.DataFrame()
    df = df.copy()
    
    # Metadata Enrichment
    if os.path.exists(metadata_path):
        with open(metadata_path, 'r') as f: metadata = json.load(f)
        df['region'] = df['country'].map(lambda x: metadata.get(x, {}).get('region', 'Other'))
        df['continent'] = df['country'].map(lambda x: metadata.get(x, {}).get('continent', 'Other'))
        df['is_nato'] = df['country'].map(lambda x: metadata.get(x, {}).get('is_nato', False))
    
    # Robust column detection for KPI calculation
    air = df.get('aircraft_total', 0)
    land = df.get('tanks', 0)
    sea = df.get('total_naval_assets', 0)
    
    # Manpower handling with safety against AttributeError
    manpower_key = 'total_manpower' if 'total_manpower' in df.columns else 'available_manpower'
    manpower = df.get(manpower_key, 1)
    if not isinstance(manpower, pd.Series): manpower = pd.Series([manpower] * len(df))
    manpower = manpower.replace(0, 1)
    
    # KPI 1: Assets per Capita (Official Project Formula)
    df['assets_per_capita'] = (air + land + sea) / manpower
    
    # KPI 2: Budget-to-GDP
    gdp = df.get('purchasing_power_parity', 1)
    if not isinstance(gdp, pd.Series): gdp = pd.Series([gdp] * len(df))
    df['budget_to_gdp_ratio'] = df.get('defense_budget', 0) / gdp.replace(0, 1)
    
    df['rank_gap'] = df.index
    return df.fillna(0)

if 'df_cleaned' in locals() and not df_cleaned.empty:
    df_final = calculate_official_kpis(df_cleaned, "country_metadata.json")
    print("Strategic KPI engineering successfully completed.")
    display_cols = ['country', 'region', 'assets_per_capita', 'budget_to_gdp_ratio']
    print(df_final[[c for c in display_cols if c in df_final.columns]].head())
else:
    print("Notice: KPI calculation skipped.")

Takeaway : The code produces a finalized analytical dataset for all countries including strategic ratios and regional labels.

---

## Dashboard Planning and Data Modeling

### Purpose
Mapping attributes to interactive Dashboard Visuals.

| Dashboard Module | Primary Metrics | Key Visualizations |
| :--- | :--- | :--- |
| Quick Stats | `power_index_rank`, `region` | Choropleth Map, Top-10 Bar Chart |
| Nation Overview | `defense_budget`, `available_manpower` | Asset Radar Charts, Gauge KPIs |
| Compare Powers | `rank_gap`, `assets_per_capita` | Side-by-Side Comparison Columns |
| Coalition Builder | Aggregate `tanks`, `aircraft_total` | Multi-select summation cards |

Takeaway : A logical architectural map for the four mandatory dashboard modules is produced to guide visual development.

---

<h2 style="color: #117A65;">Final Integration and Documentation</h2>

Finalization of the automated data pipeline.

### Export of the Master Analytical Dataset

#### Purpose
Persisting the enriched dataset for cross-platform deployment.

In [ ]:
if 'df_final' in locals() and not df_final.empty:
    os.makedirs("data/final", exist_ok=True)
    df_final.to_csv("data/final/military_final.csv", index=False)
    print("Pipeline Integration: Verified. Final analytical file persisted to data/final/military_final.csv")
else:
    print("Notice: Final export skipped.")

Takeaway : Confirmation that the project dataset with all 145 nations is ready and exported for visualization software.

<h1 style="color: #2E86C1; text-align: center;">Process Complete</h1>